# 06 · Registro del equipo y las tres zonas de I2

Se ejecuta en **Google Colab** (GPU T4), de arriba abajo. Sirve para **cualquier número de personas**: lee una carpeta por persona.

**Qué hace.** Registra a cada persona por embeddings (no hay nada que reentrenar), la evalúa dejando una sesión fuera, y mide a dónde cae cada tipo de rostro
con la regla I2 de tres zonas: **RECONOCIDO** / **DUDOSO** (decide el guardia) / **DESCONOCIDO**. Compara los umbrales de la regla (0,70 y 0,50) con los
calibrados con extraños de LFW. Tres tipos de rostro:

- **Propios** (sesión dejada fuera): deberían ser RECONOCIDOS, y nunca reconocidos como *otra* persona.
- **Compañeros no registrados** (el «amigo parecido»): no deberían ser RECONOCIDOS; lo ideal es DUDOSO o DESCONOCIDO.
- **Extraños de LFW** (personas distintas a las de calibración): deberían ser DESCONOCIDOS.

Antes de subir: genera el ZIP de cada persona con `python -m ml.src.preparar_fotos` (sesiones por fecha, 1280 px, sin metadatos). Sin
`consentimiento.json` el cargador se niega a leer a esa persona (I1). Las fotos y los embeddings **no** se suben a git (I6).

In [ ]:
import os, zipfile
from google.colab import files

subidos = files.upload()     # ml.zip + un zip por persona (luis.zip, aldana.zip, erick.zip...)
os.makedirs("/content/condohub-ia", exist_ok=True)
propios_dir = "/content/condohub-ml/datos/propios"
os.makedirs(propios_dir, exist_ok=True)
for nombre in subidos:       # Colab renombra a "ml (1).zip" si ya existía: se reconoce por empezar con «ml»
    z = zipfile.ZipFile(nombre)
    if nombre.lower().startswith("ml"):
        z.extractall("/content/condohub-ia")
        continue
    persona = os.path.splitext(nombre)[0].split(" (")[0].lower()       # luis.zip -> luis ; "aldana (1).zip" -> aldana
    con_raiz = all(n.startswith(persona + "/") for n in z.namelist())  # aldana.zip trae su carpeta; luis.zip no
    z.extractall(propios_dir if con_raiz else os.path.join(propios_dir, persona))
os.environ["ML_DATA_DIR"] = "/content/condohub-ml"
os.chdir("/content/condohub-ia")
print("personas encontradas:", sorted(os.listdir(propios_dir)))

In [ ]:
!pip install -q facenet-pytorch --no-deps
!pip install -q scikit-learn pyyaml matplotlib pillow
import torch
print("GPU disponible:", torch.cuda.is_available())

## 1 · Fotos y consentimiento (I1)

In [ ]:
import json, yaml, numpy as np
from collections import Counter
import matplotlib.pyplot as plt
from ml.src import embeddings, lfw, propios as fotos_propias, registro, zonas
from ml.src.paths import dir_datos, dir_experimento

cfg = yaml.safe_load(open("ml/configs/registro_equipo.yaml", encoding="utf-8"))
raiz = dir_datos() / cfg["carpeta_propios"]
fotos = fotos_propias.listar_fotos(raiz)      # lanza SinConsentimiento si a alguien le falta el consentimiento.json

resumen_personas = {}
for persona in sorted({f.persona for f in fotos}):
    consentimiento = fotos_propias.leer_consentimiento(raiz / persona)
    por_sesion = Counter(f.sesion for f in fotos if f.persona == persona)
    resumen_personas[persona] = {"consentimiento": consentimiento, "fotos_por_sesion": dict(por_sesion)}
    print(f"{persona}: consentimiento {consentimiento} | {sum(por_sesion.values())} fotos en {len(por_sesion)} sesiones")
    for sesion, n in sorted(por_sesion.items()):
        print(f"    {sesion}: {n}")

## 2 · Embeddings de las fotos del equipo

In [ ]:
embeber = embeddings.crear_embebedor()
imagenes = [fotos_propias.cargar_imagen(f.ruta) for f in fotos]
emb, detectado = embeber(imagenes)

personas = np.array([f.persona for f in fotos])
sesiones = np.array([f.sesion for f in fotos])
for persona in resumen_personas:
    for sesion in sorted(set(sesiones[personas == persona])):
        m = (personas == persona) & (sesiones == sesion)
        resumen_personas[persona].setdefault("rostro_detectado", {})[sesion] = f"{int(detectado[m].sum())}/{int(m.sum())}"
    print(persona, resumen_personas[persona]["rostro_detectado"])
emb, personas, sesiones = emb[detectado], personas[detectado], sesiones[detectado]
print("fotos usables:", len(emb))
del imagenes

## 3 · Extraños de LFW (para calibrar y medir)

Pocas fotos por persona alcanzan: lo que importa es cuántos extraños *distintos* hay.

In [ ]:
img_lfw, et_lfw, _ = lfw.cargar_lfw(cfg["lfw_min_fotos"])
rng = np.random.default_rng(cfg["semilla"])
elegidas = np.concatenate([rng.permutation(np.flatnonzero(et_lfw == p))[:cfg["lfw_fotos_por_persona"]]
                           for p in np.unique(et_lfw)])
emb_lfw, det_lfw = embeber([img_lfw[i] for i in elegidas])
emb_lfw, et_lfw = emb_lfw[det_lfw], et_lfw[elegidas][det_lfw]
del img_lfw
print(f"{len(emb_lfw)} fotos de {len(np.unique(et_lfw))} extraños")

## 4 · Las tres zonas

In [ ]:
puntos = zonas.puntajes_equipo(emb, personas, sesiones, emb_lfw, et_lfw, cfg["semilla"])
regla = (cfg["umbrales_regla"]["alto"], cfg["umbrales_regla"]["bajo"])
res = zonas.evaluar_zonas(puntos, {"regla_inicial": regla}, cfg["far_alto"], cfg["far_bajo"])

def pct(x):
    return f"{100 * x:5.1f} %"

for nombre, r in res.items():
    print(f"\n=== {nombre}: alto = {r['alto']:.3f}, bajo = {r['bajo']:.3f}")
    g = r["genuinos"]
    if g:
        print(f"  Propios ({g['fotos']} fotos, sesión dejada fuera):")
        print(f"    RECONOCIDO correcto {pct(g['reconocido_correcto'])} | RECONOCIDO como OTRA persona {pct(g['reconocido_otra_persona'])}")
        print(f"    DUDOSO con candidato correcto {pct(g['dudoso_candidato_correcto'])} | DUDOSO con candidato otro {pct(g['dudoso_candidato_otra_persona'])}")
        print(f"    DESCONOCIDO (no los reconoce) {pct(g['desconocido'])}")
    else:
        print("  Propios: ninguna persona tiene 2 o más sesiones")
    c = r["companeros_no_registrados"]
    if c:
        print(f"  Compañero NO registrado -> RECONOCIDO {pct(c['RECONOCIDO'])} | DUDOSO {pct(c['DUDOSO'])} | DESCONOCIDO {pct(c['DESCONOCIDO'])}")
    e = r["extranos_prueba"]
    print(f"  Extraños de LFW (prueba) -> RECONOCIDO {pct(e['RECONOCIDO'])} | DUDOSO {pct(e['DUDOSO'])} | DESCONOCIDO {pct(e['DESCONOCIDO'])}")
print("\nsesiones omitidas por ser la única de la persona:", list(puntos["sesiones_omitidas"]))

In [ ]:
base = registro.evaluar(emb, personas, sesiones, emb_lfw, et_lfw,
                        {"semilla": cfg["semilla"], "far_maximo": cfg["far_alto"]})
print("umbral único calibrado (FAR 1 %):", round(base["umbral_calibrado_con_impostores_de_validacion"], 3),
      "| FAR en prueba:", round(base["far_prueba"], 4), "| FRR:", base["frr"], "| confusiones:", base["confusiones"])
for fila in base["por_sesion"]:
    print(fila)

In [ ]:
destino = dir_experimento(cfg["nombre"])
fig, ax = plt.subplots(figsize=(8, 4.5))
for datos, etiqueta, color in ((puntos["genuinos"], "propios (sesión fuera)", "tab:green"),
                               (puntos["companeros"], "compañeros no registrados", "tab:orange"),
                               (puntos["impostores_prueba"], "extraños de LFW", "tab:gray")):
    if len(datos):
        ax.hist(datos, bins=30, range=(0, 1), alpha=0.55, label=etiqueta, color=color, density=True)
ax.axvline(regla[0], color="k", ls="--", label=f"regla: alto {regla[0]}")
ax.axvline(regla[1], color="k", ls=":", label=f"regla: bajo {regla[1]}")
cal = res["calibrado_con_extranos"]
ax.axvline(cal["alto"], color="red", ls="--", label=f"calibrado: alto {cal['alto']:.2f}")
ax.axvline(cal["bajo"], color="red", ls=":", label=f"calibrado: bajo {cal['bajo']:.2f}")
ax.set_xlabel("similitud coseno con el perfil más parecido"); ax.set_ylabel("densidad"); ax.legend(fontsize=7)
fig.tight_layout(); fig.savefig(destino / "histograma_similitudes.png", dpi=150); plt.show()

## 5 · Guardar y descargar

Solo métricas y una figura (sin fotos ni embeddings, I1/I6). Se copian a `ml/experiments/<fecha>_registro_equipo/`.

In [ ]:
from google.colab import files

resultado = {
    "config": cfg, "personas": resumen_personas, "fotos_usables": int(len(emb)),
    "extranos_lfw": {"fotos": int(len(emb_lfw)), "personas": int(len(np.unique(et_lfw)))},
    "zonas": res, "registro_umbral_unico": base,
    "sesiones_omitidas": [str(s) for s in puntos["sesiones_omitidas"]],
}
(destino / "metrics.json").write_text(
    json.dumps(resultado, indent=2, ensure_ascii=False, default=lambda o: o.item() if hasattr(o, "item") else str(o)),
    encoding="utf-8")
for nombre in ("metrics.json", "histograma_similitudes.png"):
    files.download(str(destino / nombre))